# 04 · Legambiente: closed ski facilities in Italy (PDF → table)

**Owner:** Luca · **Sub-question:** SQ5 (Do closed facilities fall into the "at risk" snow group?)

**Source:** Legambiente, *Nevediversa 2026* (March 2026), 359 pages, PDF.  
Download: https://www.legambiente.it/wp-content/uploads/2026/03/NeveDiversa2026.pdf  
Save it as `data/raw/NeveDiversa2026.pdf` (the `raw` folder is not pushed to GitHub).

**What this notebook does**
1. Reads the two lists of the report: *Impianti dismessi* (pages 14–40) and *Impianti temporaneamente chiusi* (pages 50–56).
2. Rebuilds the reading order of the 3-column layout, then extracts one row per site: name, municipality, province, region, altitude, type, years.
3. Saves `data/processed/legambiente_closures.csv` and prints the record counts for the integration log.

**Not done here:** geocoding (coordinates via Nominatim) comes in the next step; the columns `lat`, `lon`, `matched` of the data dictionary are filled there.

In [1]:
import re
import csv
from pathlib import Path
from collections import Counter

import pdfplumber
import pandas as pd

# Find the repository root (the folder that contains requirements.txt), so the paths work from any folder
ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

PDF_PATH = ROOT / "data" / "raw" / "NeveDiversa2026.pdf"
OUT_PATH = ROOT / "data" / "processed" / "legambiente_closures.csv"

# Page ranges of the two lists in the report (printed page = PDF page)
PAGES_DISMESSI = (14, 40)       # Impianti dismessi (abandoned)
PAGES_CHIUSI = (50, 56)         # Impianti temporaneamente chiusi (temporarily closed)

print("Repository root:", ROOT)
print("PDF found:", PDF_PATH.exists())

Repository root: /home/claude/leg/test_repo
PDF found: True


## 1 · Reading order of the PDF

Each page has three text columns, and one entry (name, *Comune*, *Quota*, *Tipologia*, years, description) often continues in the next column or page. `pdfplumber.extract_text()` on the full page would mix the columns, so each page is cut into three vertical strips (left, middle, right), read top to bottom, and the strips are joined in order. Page numbers, running headers and footers are removed.

In [2]:
# x-ranges (in PDF points) of the three columns; the page is 595 points wide
COLUMNS = ((60, 232), (232, 393), (393, 570))
TOP, BOTTOM = 40, 825     # cut off running header and footer

NOISE = re.compile(r"^(RAPPORTO DI LEGAMBIENTE|NEVEDIVERSA 2026|ANALISI DEGLI IMPIANTI E DEL TERRITORIO|IM ?PIA ?N ?TI.*|\d{1,3})$")
LABELS = {"Comune", "Quota", "Tipologia", "Anno di costruzione", "Anno di dismissione", "Anno di chiusura"}

REGIONS = {
    "valle d’aosta": "Valle d’Aosta", "piemonte": "Piemonte", "liguria": "Liguria", "lombardia": "Lombardia",
    "trentino-alto adige": "Trentino-Alto Adige", "veneto": "Veneto", "friuli venezia giulia": "Friuli Venezia Giulia",
    "emilia-romagna": "Emilia-Romagna", "toscana": "Toscana", "marche": "Marche", "lazio": "Lazio",
    "abruzzo": "Abruzzo", "campania": "Campania", "molise": "Molise", "basilicata": "Basilicata",
    "calabria": "Calabria", "sicilia": "Sicilia", "sardegna": "Sardegna", "umbria": "Umbria",
}


def read_lines(pdf, first_page, last_page):
    """Return a list of (page, line) in reading order: column by column, page by page."""
    out = []
    for page_number in range(first_page, last_page + 1):
        page = pdf.pages[page_number - 1]
        for x0, x1 in COLUMNS:
            text = page.crop((x0, TOP, x1, BOTTOM)).extract_text() or ""
            for line in text.split("\n"):
                line = line.strip()
                if line and not NOISE.match(line):
                    out.append((page_number, line))
    return out

## 2 · From lines to rows

The report repeats the same block for every site:

```
<Region>
numero impianti: N
<Name>
Comune
<Municipality (province)>
Quota
<altitude>
Tipologia
<type>
Anno di costruzione / Anno di dismissione (or chiusura)
<years>
<free-text description>
```

The label **`Comune`** marks the start of one site, so there is one row per `Comune` line. The region is the line just before the most recent `numero impianti: N`. The name is the line (or lines) before `Comune`.

In [3]:
def parse_sites(lines, status):
    """Turn the ordered lines into one dict per site."""
    texts = [line for _, line in lines]
    pages = [page for page, _ in lines]

    # Region of every line = the heading before the latest "numero impianti"
    current, region_of = None, []
    for i, line in enumerate(texts):
        if line.startswith("numero impianti") and i > 0:
            heading = texts[i - 1].strip()
            current = REGIONS.get(heading.lower(), heading)
        region_of.append(current)

    rows = []
    for i, line in enumerate(texts):
        if line != "Comune":
            continue

        # Municipality (up to 3 lines, until the next label)
        j, comune_lines = i + 1, []
        while j < len(texts) and texts[j] not in LABELS and not texts[j].startswith("numero impianti") and len(comune_lines) < 3:
            comune_lines.append(texts[j])
            j += 1
        comune = re.sub(r"\s+A cura di.*$", "", " ".join(comune_lines))
        match = re.search(r"\(([A-Z]{2,3})\)\s*$", comune)
        province = match.group(1) if match else ""
        comune = re.sub(r"\s*\([A-Z]{2,3}\)\s*$", "", comune)

        # Fields after the municipality
        fields, k = {}, j
        while k < len(texts) and texts[k] in LABELS:
            label, k, values, stopped = texts[k], k + 1, [], False
            while k < len(texts) and len(values) < 4:
                if texts[k] in LABELS:
                    stopped = True
                    break
                values.append(texts[k])
                k += 1
            if label.startswith("Anno"):
                fields[label] = values[0] if values else ""        # only the first line is the year
            elif label == "Tipologia":
                fields[label] = " ".join(values) if stopped else (values[0] if values else "")
            else:
                fields[label] = " ".join(values[:3]) if stopped else (values[0] if values else "")
            if label in ("Anno di dismissione", "Anno di chiusura") or not stopped:
                break

        # Name: lines before "Comune"; names usually start with "Impianto/Impianti (loc.)"
        name = None
        for back in range(1, 4):
            if i - back >= 0 and re.match(r"^Impiant[oi]\b", texts[i - back]):
                name = " ".join(texts[i - back:i])
                break
        if name is None:
            name = texts[i - 1]
        name = re.sub(r"^Impiant[oi]\s+(loc\.|Loc\.)?\s*", "", name).strip() or name

        rows.append({
            "facility_name": re.sub(r"\s+", " ", name),
            "comune": comune,
            "province": province,
            "region": region_of[i],
            "status": status,
            "altitude_text": fields.get("Quota", ""),
            "facility_type": fields.get("Tipologia", ""),
            "year_built": fields.get("Anno di costruzione", ""),
            "year_closed": fields.get("Anno di dismissione") or fields.get("Anno di chiusura", ""),
            "source_page": pages[i],
        })
    return rows

In [4]:
with pdfplumber.open(PDF_PATH) as pdf:
    dismessi = parse_sites(read_lines(pdf, *PAGES_DISMESSI), "dismesso")
    chiusi = parse_sites(read_lines(pdf, *PAGES_CHIUSI), "chiuso temporaneamente")

closures = pd.DataFrame(dismessi + chiusi)
closures.to_csv(OUT_PATH, index=False, encoding="utf-8")

print("Sites extracted (dismesso):               ", len(dismessi))
print("Sites extracted (chiuso temporaneamente): ", len(chiusi))
print("Total rows saved:                         ", len(closures))
print("Saved to:", OUT_PATH.relative_to(ROOT))
closures.head()

Sites extracted (dismesso):                147
Sites extracted (chiuso temporaneamente):  37
Total rows saved:                          184
Saved to: data/processed/legambiente_closures.csv


,facility_name,comune,province,region,status,altitude_text,facility_type,year_built,year_closed,source_page
0,Challand - Saint Anselme,Challand-Saint Anselme,AO,Valle d’Aosta,dismesso,730 s.l.m. m,uno skilift,inizio anni ‘80,fine anni ‘90,14
1,Ozein,Aymavilles,AO,Valle d’Aosta,dismesso,1400 m s.l.m - 1650 m s.l.m,uno skilift,anni ‘60,1973,14
2,Breuil-Cima del Furggen,Valtournenche,AO,Valle d’Aosta,dismesso,2400 m s.l.m. - 3.486 m s.l.m.,Funivia e annesso tunnel di cemento per uscita...,1952,1993,14
3,Clos,La Magdaleine,AO,Valle d’Aosta,dismesso,1660 m s.l.m. - 1820 m s.l.m.,uno skilift,1952,1993,14
4,Breuil-Cervinia,Valtournenche,AO,Valle d’Aosta,dismesso,2900 m s.l.m circa,"quattro funivie, uno skilift",anni ‘60,inizio anni ’90,14


## 3 · Checks

- **Completeness:** the number of rows must equal the number of `Comune` labels in the PDF (one per site).
- **Regions:** every row has a region; the regions with sites must match the regional headings of the report.
- **What is a "site"?** The report counts *lifts* (273 dismissed in 2026), but one site can have several lifts (for example "four cableways and one ski lift"). Our table has one row per **site**, so the row count is lower than 273 by design.

In [5]:
# Independent count: number of "Comune" labels (word at a column start) on the same pages, without cutting columns
def count_comune_labels(pdf, first_page, last_page):
    total = 0
    for page_number in range(first_page, last_page + 1):
        for word in pdf.pages[page_number - 1].extract_words():
            if word["text"] == "Comune" and min(abs(word["x0"] - x) for x in (74, 234, 395)) < 4:
                total += 1
    return total

with pdfplumber.open(PDF_PATH) as pdf:
    labels_dismessi = count_comune_labels(pdf, *PAGES_DISMESSI)
    labels_chiusi = count_comune_labels(pdf, *PAGES_CHIUSI)

print("Comune labels in the PDF (dismesso):", labels_dismessi, "| rows:", len(dismessi))
print("Comune labels in the PDF (chiuso):  ", labels_chiusi, "| rows:", len(chiusi))
print("(a small difference is expected: the word 'Comune' also appears inside the free-text descriptions)")
print()
print("Rows without region:  ", closures["region"].isna().sum())
print("Rows without province:", (closures["province"] == "").sum())
print()
print(closures.groupby(["status", "region"]).size().unstack(0).fillna(0).astype(int))

Comune labels in the PDF (dismesso): 149 | rows: 147
Comune labels in the PDF (chiuso):   38 | rows: 37
(a small difference is expected: the word 'Comune' also appears inside the free-text descriptions)

Rows without region:   0
Rows without province: 2

status                 chiuso temporaneamente  dismesso
region                                                 
Abruzzo                                     5        10
Basilicata                                  2         2
Calabria                                    1         2
Campania                                    0         3
Emilia-Romagna                              6        12
Friuli Venezia Giulia                       1         7
Lazio                                       3         2
Liguria                                     1         1
Lombardia                                   2        32
Marche                                      1         3
Molise                                      2         1
Piemonte         

## 4 · Notes for the integration log and limitations

- **Source → table:** PDF with 359 pages → 184 site rows (147 dismesso + 37 chiuso temporaneamente). No site was removed; rows that look similar (for example 3 ski lifts in the "Conca del Sole" area in Pragelato) differ in altitude and years, so they are separate lifts.
- **Province missing** for two rows because the report does not print it; they can be completed during geocoding.
- **Free text:** `year_built`, `year_closed`, `altitude_text` and `facility_type` are kept as written in the report (for example "anni ‘60", "2011 (?)", "nd"). They are not cleaned because the analysis (SQ5) only needs the location and the status.
- **Scope:** the report covers the Alps and the Apennines. Only sites that can be matched to a snow-risk class (after geocoding) enter the chi-squared test; the number of unmatched names must be documented.
- **Next step:** geocode `comune` + `province` with Nominatim (cache the answers, at most one request per second), then add `lat`, `lon`, `matched`.